In [11]:
import sys
sys.path.append('../')

import os
import re
import json
import click
import torch
import dnnlib
from torch_utils import distributed as dist
from training import training_loop

import matplotlib.pyplot as plt
import h5py
import numpy as np

from torch.utils.data import Dataset

sys.path.insert(0, '/home/trevor/trevor/repos/3D_SR_CT/data')
import augmentation


In [12]:
class PatchDataset(Dataset):
    def __init__(self, hr_dir, if_translation=True, if_rotation=True, if_flip=True, if_downsample=True, flip_prob=0.5, tgt_patch_size=(48, 48, 48), curr_patch_size=(64, 64, 64)):
        self.hr_dir = hr_dir
        # store full patch paths
        self.hr_paths = []
        # store corresponding image names
        self.hr_img_names = []

        for image_folder in os.listdir(hr_dir):
            image_path = os.path.join(hr_dir, image_folder)

            # Ensure it's a directory
            if os.path.isdir(image_path):
                for patch in os.listdir(image_path):
                    patch_path = os.path.join(image_path, patch)

                    if os.path.isfile(patch_path):
                        self.hr_paths.append(patch_path)
                        self.hr_img_names.append(image_folder)
        
        # augmentation args
        self.translation = if_translation
        self.rotation = if_rotation
        self.flip = if_flip
        self.downsample = if_downsample
        self.flip_prob = flip_prob
        self.tgt_patch_size = tgt_patch_size
        self.curr_patch_size = curr_patch_size

    def __len__(self):
        return len(self.hr_paths)

    def augment(self, hr_patch):
        aug_patch = hr_patch
        if self.translation:
            aug_patch = augmentation.translation(aug_patch, self.tgt_patch_size, self.curr_patch_size)
        if self.rotation:
            aug_patch = augmentation.rotation(aug_patch)
        if self.flip:
            aug_patch = augmentation.flip(aug_patch, self.flip_prob)
        if self.downsample:
            aug_patch = augmentation.downsample(aug_patch)
        return aug_patch

    def __getitem__(self, index):
        hr_patch = np.load(self.hr_paths[index])
        aug_patch = self.augment(hr_patch)
        
        return torch.tensor(hr_patch, dtype = torch.float32), torch.tensor(aug_patch, dtype = torch.float32)



In [14]:
# read patches from folder 'edsrct_patches'
directory = "/home/trevor/trevor/repos/3D_SR_CT/data/3dsrct_patches"
print(directory)
train_data = PatchDataset(hr_dir = directory)


/home/trevor/trevor/repos/3D_SR_CT/data/3dsrct_patches


In [16]:
print("Dataset length:", len(train_data))

Dataset length: 64355


- train_data (after loading with PatchDataset) now has pairs of hr, augmented patches
- what is train label?
- During training, what's the input and output of the model?